# Sumativa 1: Análisis Exploratorio e Inferencia Estadística
**Magíster en Ciencia de Datos e Inteligencia Artificial - UNAB**
- **Integrantes:** Verónica Durán, Raúl Moya, Daniela Rojas, Manuel Sánchez
- **Grupo 2**
- **Curso:** MCDI501 - Estadística Computacional para la Toma de Decisiones
- **Dataset:** Online Shoppers Purchasing Intention



### Criterio metodológico de esta versión

Esta versión conserva los cálculos reproducibles de la exploración inicial y fortalece su interpretación estadística. La lectura de los resultados sigue la secuencia **contexto → tipo de variable → descripción → relación → inferencia → interpretación**, evitando atribuir causalidad a asociaciones observacionales.


## 1. Contexto y descripción del conjunto de datos

El conjunto de datos **Online Shoppers Purchasing Intention** contiene información de **12.330 sesiones** de navegación en un sitio de comercio electrónico. Combina variables relacionadas con páginas visitadas, duración de navegación, métricas de Google Analytics y características de la sesión.

La variable de resultado es **`Revenue`**, que indica si la sesión terminó (`True`) o no (`False`) en una compra.

**Fuente:** UCI Machine Learning Repository — *Online Shoppers Purchasing Intention Dataset*.

En esta etapa se conserva el archivo original sin modificaciones en `data/raw/`.


## 2. Importación de librerías


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

## 3. Carga reproducible de los datos

El bloque siguiente permite ejecutar el notebook tanto desde la carpeta `notebooks/` como desde la raíz del repositorio.


In [ ]:
rutas_posibles = [
    Path("../data/raw/online_shoppers_intention.csv"),
    Path("data/raw/online_shoppers_intention.csv"),
]

ruta = next((p for p in rutas_posibles if p.exists()), None)

if ruta is None:
    raise FileNotFoundError(
        "No se encontró 'online_shoppers_intention.csv'. "
        "Verifique que esté en data/raw/."
    )

df = pd.read_csv(ruta)

print(f"Archivo cargado: {ruta.resolve()}")
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()


## 4. Reconocimiento inicial de la base

Antes de calcular estadísticas se revisan dimensiones, nombres de columnas, tipos computacionales, valores faltantes, duplicados y categorías. Esta etapa permite verificar que la estructura del archivo corresponda con lo esperado y evita aplicar procedimientos estadísticos a variables mal clasificadas.


In [ ]:
print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())


In [ ]:
df.info()


In [ ]:
resumen_calidad = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_no_nulos": df.notna().sum(),
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2),
    "n_unicos": df.nunique(dropna=False)
})

resumen_calidad


In [ ]:
duplicados = df.duplicated().sum()

print(f"Filas duplicadas exactas: {duplicados}")
print(f"Porcentaje de duplicados: {duplicados / len(df) * 100:.2f}%")


### Calidad de los datos y registros duplicados

El conjunto contiene **12.330 sesiones y 18 variables**, sin valores faltantes. Se identificaron **125 filas exactamente duplicadas (1,01 %)**. Estos registros se mantienen en este estado porque el dataset no dispone de un identificador único de sesión que permita demostrar que corresponden a errores de duplicación y no a sesiones distintas con iguales características observadas.

La decisión metodológica es, por tanto, **documentar los duplicados y no eliminarlos automáticamente**. Esto evita modificar observaciones potencialmente válidas sin evidencia suficiente.


## 5. Identificación y clasificación estadística de las variables

La clasificación estadística no depende únicamente del `dtype` de pandas. Por ejemplo, una variable almacenada como entero puede representar una categoría codificada y, por tanto, ser cualitativa nominal.

La siguiente tabla documenta la clasificación utilizada en el proyecto.


In [ ]:
clasificacion = pd.DataFrame([
    ["Administrative", "Cuantitativa", "Discreta", "Número de páginas administrativas visitadas"],
    ["Administrative_Duration", "Cuantitativa", "Continua", "Tiempo dedicado a páginas administrativas"],
    ["Informational", "Cuantitativa", "Discreta", "Número de páginas informativas visitadas"],
    ["Informational_Duration", "Cuantitativa", "Continua", "Tiempo dedicado a páginas informativas"],
    ["ProductRelated", "Cuantitativa", "Discreta", "Número de páginas relacionadas con productos visitadas"],
    ["ProductRelated_Duration", "Cuantitativa", "Continua", "Tiempo dedicado a páginas de productos"],
    ["BounceRates", "Cuantitativa", "Continua", "Tasa de rebote"],
    ["ExitRates", "Cuantitativa", "Continua", "Tasa de salida"],
    ["PageValues", "Cuantitativa", "Continua", "Valor de página asociado a la sesión"],
    ["SpecialDay", "Cuantitativa", "Continua", "Cercanía de la sesión a una fecha especial"],
    ["Month", "Cualitativa", "Ordinal/temporal", "Mes de la sesión"],
    ["OperatingSystems", "Cualitativa", "Nominal", "Sistema operativo codificado"],
    ["Browser", "Cualitativa", "Nominal", "Navegador codificado"],
    ["Region", "Cualitativa", "Nominal", "Región codificada"],
    ["TrafficType", "Cualitativa", "Nominal", "Tipo/fuente de tráfico codificado"],
    ["VisitorType", "Cualitativa", "Nominal", "Tipo de visitante"],
    ["Weekend", "Cualitativa", "Nominal dicotómica", "Indica si la sesión ocurrió en fin de semana"],
    ["Revenue", "Cualitativa", "Nominal dicotómica", "Indica si la sesión terminó en compra"],
], columns=["Variable", "Naturaleza", "Tipo", "Descripción"])

clasificacion


### Interpretación de la clasificación de variables

La clasificación estadística se realiza según el **significado de cada variable**, no únicamente según el `dtype` asignado por pandas. Por ejemplo, `OperatingSystems`, `Browser`, `Region` y `TrafficType` aparecen codificadas con números, pero esos números representan categorías y no cantidades sobre las que tenga sentido calcular una media.

`Month` se considera una **variable cualitativa temporal/cíclica**: sus categorías representan meses y su interpretación está asociada a patrones temporales, pero no constituye una escala ordinal convencional de distancias comparables.

`SpecialDay` corresponde a un **índice numérico de proximidad a una fecha especial**. Aunque conceptualmente representa una magnitud de proximidad, en el dataset solo se observan valores discretizados. Esta particularidad se documenta para evitar interpretarlo como una variable continua ordinaria sin matices.

La variable objetivo `Revenue` es **cualitativa nominal dicotómica** (`True`/`False`) e indica si la sesión finalizó o no en una compra.


## 6. Estadística descriptiva

### 6.1 Variables cuantitativas

Para las variables cuantitativas se calculan medidas de tendencia central y dispersión. Se incluyen media y mediana porque una diferencia importante entre ambas puede sugerir asimetría. La desviación estándar y el rango intercuartílico permiten describir la dispersión.


In [ ]:
variables_cuantitativas = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
]

resumen_num = df[variables_cuantitativas].describe().T
resumen_num["mediana"] = df[variables_cuantitativas].median()
resumen_num["varianza"] = df[variables_cuantitativas].var()
resumen_num["IQR"] = (
    df[variables_cuantitativas].quantile(0.75)
    - df[variables_cuantitativas].quantile(0.25)
)
resumen_num["CV_%"] = (
    df[variables_cuantitativas].std()
    / df[variables_cuantitativas].mean().replace(0, np.nan)
    * 100
)
resumen_num["asimetria"] = df[variables_cuantitativas].skew()
resumen_num["curtosis"] = df[variables_cuantitativas].kurt()

resumen_num.round(3)


### 6.2 Variables cualitativas

Para las variables categóricas son más pertinentes las frecuencias y proporciones que la media.


In [ ]:
variables_categoricas = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
    "Revenue",
]

for col in variables_categoricas:
    tabla = pd.DataFrame({
        "Frecuencia": df[col].value_counts(dropna=False),
        "Porcentaje": (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)
    })
    print(f"\n--- {col} ---")
    print(tabla)


## 7. Distribución de la variable objetivo: Revenue

Antes de estudiar relaciones con otras variables se caracteriza el resultado principal. Esto también permite identificar si existe desbalance entre sesiones con compra y sin compra.


In [ ]:
revenue_resumen = pd.DataFrame({
    "Frecuencia": df["Revenue"].value_counts(),
    "Porcentaje": (df["Revenue"].value_counts(normalize=True) * 100).round(2)
})
revenue_resumen


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Revenue")
plt.title("Distribución de sesiones según Revenue")
plt.xlabel("Sesión terminó en compra")
plt.ylabel("Número de sesiones")
plt.tight_layout()
plt.show()


### Interpretación de la variable objetivo `Revenue`

De las **12.330 sesiones** analizadas, **1.908 finalizaron en compra (15,47 %)**, mientras que **10.422 no finalizaron en compra (84,53 %)**. Esto evidencia un **desbalance de clases**, ya que la gran mayoría de las sesiones corresponde a la categoría sin compra.

Este resultado es relevante para la comprensión del problema y deberá considerarse en etapas posteriores del proyecto, especialmente si se desarrollan modelos predictivos. En esta etapa, se utiliza principalmente para caracterizar la distribución de la variable objetivo.

## 8. Visualización de variables cuantitativas

Se seleccionan variables especialmente relacionadas con el comportamiento de navegación. Los histogramas permiten observar forma, concentración y asimetría; los boxplots permiten comparar distribuciones según `Revenue`.


In [ ]:
variables_graficos = [
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
]

for col in variables_graficos:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, bins=30, kde=True)
    plt.title(f"Distribución de {col}")
    plt.tight_layout()
    plt.show()


In [ ]:
for col in variables_graficos:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x="Revenue", y=col)
    plt.title(f"{col} según resultado de compra")
    plt.xlabel("Revenue")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 9. Relaciones preliminares entre variables

### 9.1 Relación entre variables cuantitativas

La matriz de correlación permite identificar asociaciones lineales entre variables cuantitativas. Una correlación alta no implica causalidad y también puede advertir que dos variables contienen información similar.


In [ ]:
corr = df[variables_cuantitativas].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación de variables cuantitativas")
plt.tight_layout()
plt.show()


In [ ]:
pares_corr = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .sort_values(key=abs, ascending=False)
        .rename("correlacion")
        .reset_index()
        .rename(columns={"level_0": "Variable_1", "level_1": "Variable_2"})
)

pares_corr.head(10)


### 9.2 Variables categóricas y Revenue

Las tablas de contingencia permiten comparar la proporción de compra entre categorías. A continuación se exploran `VisitorType`, `Weekend` y `Month`.


In [ ]:
for col in ["VisitorType", "Weekend", "Month"]:
    tabla = pd.crosstab(df[col], df["Revenue"], margins=True)
    print(f"\nTabla de contingencia: {col} vs Revenue")
    print(tabla)

    prop = pd.crosstab(df[col], df["Revenue"], normalize="index") * 100
    print("\nPorcentajes por categoría:")
    print(prop.round(2))


In [ ]:
for col in ["VisitorType", "Weekend", "Month"]:
    prop_compra = (
        df.groupby(col, observed=False)["Revenue"]
          .mean()
          .mul(100)
          .sort_values(ascending=False)
    )

    plt.figure(figsize=(8, 4))
    prop_compra.plot(kind="bar")
    plt.title(f"Porcentaje de sesiones con compra según {col}")
    plt.ylabel("% de sesiones con Revenue = True")
    plt.xlabel(col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 10. Estimación puntual e intervalo de confianza

Para cubrir la estimación inferencial se estima la **proporción poblacional de sesiones que terminan en compra**.

La estimación puntual es la proporción observada de `Revenue = True`. Se construye además un intervalo de confianza del 95 % para esa proporción mediante la aproximación normal.


In [ ]:
n = len(df)
x = df["Revenue"].astype(bool).sum()
p_hat = x / n

z = stats.norm.ppf(0.975)
se = np.sqrt(p_hat * (1 - p_hat) / n)

li = max(0, p_hat - z * se)
ls = min(1, p_hat + z * se)

print(f"Número de sesiones: {n:,}")
print(f"Sesiones con compra: {x:,}")
print(f"Estimación puntual de la proporción de compra: {p_hat:.4f} ({p_hat*100:.2f}%)")
print(f"IC 95% para la proporción: [{li:.4f}, {ls:.4f}]")
print(f"IC 95% en porcentaje: [{li*100:.2f}%, {ls*100:.2f}%]")


### Estimación puntual e intervalo de confianza

La proporción muestral de sesiones que finalizan en compra es:

**p̂ = 1.908 / 12.330 = 0,1547 (15,47 %).**

El intervalo de confianza del 95 % obtenido mediante aproximación normal es **[14,84 %, 16,11 %]**. Las condiciones de aproximación son ampliamente satisfechas, ya que tanto el número de compras como el de no compras son muy superiores a 10.

En términos frecuentistas, el procedimiento utilizado produciría intervalos que contienen la verdadera proporción poblacional aproximadamente en el 95 % de muestras repetidas bajo condiciones equivalentes. Para este conjunto de datos, el intervalo obtenido entrega un rango plausible para la tasa poblacional de conversión.


## 11. Prueba de hipótesis: VisitorType y Revenue

Se evaluará si existe asociación estadística entre el **tipo de visitante (`VisitorType`)** y el resultado de compra (`Revenue`).

### Hipótesis

- **H₀:** `VisitorType` y `Revenue` son independientes; no existe asociación estadística entre ambas variables.
- **H₁:** `VisitorType` y `Revenue` no son independientes; existe asociación estadística entre ambas variables.

Se utiliza una **prueba chi-cuadrado de independencia**, apropiada para analizar la relación entre dos variables categóricas.

**Nivel de significancia:** α = 0,05.


In [ ]:
tabla_contingencia = pd.crosstab(df["VisitorType"], df["Revenue"])
tabla_contingencia


In [ ]:
chi2, p_valor, gl, esperadas = stats.chi2_contingency(tabla_contingencia)

esperadas_df = pd.DataFrame(
    esperadas,
    index=tabla_contingencia.index,
    columns=tabla_contingencia.columns
)

print(f"Chi-cuadrado: {chi2:.4f}")
print(f"Grados de libertad: {gl}")
print(f"p-valor: {p_valor:.6g}")
print(f"Frecuencia esperada mínima: {esperadas.min():.2f}")

alpha = 0.05

if p_valor < alpha:
    print("\nDecisión: se rechaza H₀ al nivel de significancia del 5%.")
    print("Existe evidencia estadística de asociación entre VisitorType y Revenue.")
else:
    print("\nDecisión: no se rechaza H₀ al nivel de significancia del 5%.")
    print("No se dispone de evidencia suficiente para afirmar una asociación entre VisitorType y Revenue.")

esperadas_df


### Interpretación de la prueba chi-cuadrado

Con un nivel de significancia de **α = 0,05**, se rechaza la hipótesis nula de independencia, ya que **p < 0,001**. Por lo tanto, existe evidencia estadísticamente significativa de **asociación entre el tipo de visitante (`VisitorType`) y la realización de una compra (`Revenue`)**.

Además, la frecuencia esperada mínima es **13,15**, superior a 5, por lo que se satisface esta condición de aplicación de la prueba chi-cuadrado. Descriptivamente, los visitantes nuevos presentan una proporción de compra de **24,91 %**, mientras que los visitantes recurrentes presentan una proporción de **13,93 %**.

Este resultado indica asociación estadística, pero no permite establecer una relación causal.

## 12. Tamaño del efecto: V de Cramér

Un p-valor permite evaluar evidencia contra la hipótesis nula, pero no informa por sí solo la **magnitud** de la asociación. Por ello se calcula V de Cramér.


In [ ]:
n_chi = tabla_contingencia.to_numpy().sum()
r, k = tabla_contingencia.shape

v_cramer = np.sqrt(chi2 / (n_chi * min(r - 1, k - 1)))

print(f"V de Cramér: {v_cramer:.4f}")


### Interpretación del tamaño de efecto

El **V de Cramér = 0,105** indica que, aunque la asociación entre `VisitorType` y `Revenue` es estadísticamente significativa, su **magnitud es baja**. Por tanto, el tipo de visitante aporta información sobre la ocurrencia de compra, pero no debe interpretarse como un factor fuertemente asociado por sí solo.

### Estimación puntual e intervalo de confianza

La proporción muestral de sesiones que finalizan en compra es:

**p̂ = 1.908 / 12.330 = 0,1547 (15,47 %).**

El intervalo de confianza del 95 % obtenido mediante aproximación normal es **[14,84 %, 16,11 %]**. Las condiciones de aproximación son ampliamente satisfechas, ya que tanto el número de compras como el de no compras son muy superiores a 10.

En términos frecuentistas, el procedimiento utilizado produciría intervalos que contienen la verdadera proporción poblacional aproximadamente en el 95 % de muestras repetidas bajo condiciones equivalentes. Para este conjunto de datos, el intervalo obtenido entrega un rango plausible para la tasa poblacional de conversión.


## 13. Análisis complementario: PageValues según Revenue

Como exploración adicional se compara descriptivamente `PageValues` entre sesiones con y sin compra. Esta sección ayuda a identificar si el comportamiento de esta variable cambia según el resultado de la sesión.


In [ ]:
comparacion_pagevalues = (
    df.groupby("Revenue")["PageValues"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(3)
)

comparacion_pagevalues


In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Revenue", y="PageValues")
plt.title("PageValues según resultado de compra")
plt.xlabel("Revenue")
plt.ylabel("PageValues")
plt.tight_layout()
plt.show()


### Calidad de los datos y registros duplicados

El conjunto contiene **12.330 sesiones y 18 variables**, sin valores faltantes. Se identificaron **125 filas exactamente duplicadas (1,01 %)**. Estos registros se mantienen en esta etapa porque el dataset no dispone de un identificador único de sesión que permita demostrar que corresponden a errores de duplicación y no a sesiones distintas con iguales características observadas.

La decisión metodológica es, por tanto, **documentar los duplicados y no eliminarlos automáticamente**. Esto evita modificar observaciones potencialmente válidas sin evidencia suficiente.


## 15. Ficha descriptiva para la Evaluación Formativa 1

### Problema abordado
Comprender qué características observadas durante una sesión de navegación se relacionan con la finalización de una compra en un sitio de comercio electrónico.

### Unidad de análisis
Una sesión de navegación de un usuario.

### Variable de resultado
`Revenue`: variable cualitativa nominal dicotómica que indica si la sesión terminó o no en compra.

### Variables explicativas
El conjunto incluye variables cuantitativas relacionadas con páginas visitadas, duración, tasas de rebote/salida y valor de página; además de variables categóricas relacionadas con mes, sistema operativo, navegador, región, tráfico, tipo de visitante y fin de semana.

### Relaciones preliminares a estudiar
- Comportamiento de navegación y `Revenue`.
- `VisitorType` y `Revenue`.
- `Month` y `Revenue`.
- `Weekend` y `Revenue`.
- Relaciones entre métricas cuantitativas de navegación.

### Alcance
El análisis es exploratorio e inferencial inicial. Las asociaciones encontradas no permiten establecer relaciones causales.


### Interpretación de la estadística descriptiva

Las variables cuantitativas presentan, en general, **asimetría positiva**, con concentraciones importantes de valores bajos y una menor cantidad de observaciones elevadas. Esto se aprecia especialmente en variables de duración y en `PageValues`. En consecuencia, la media puede verse influida por valores altos, por lo que se interpreta conjuntamente con la **mediana y el rango intercuartílico (IQR)**.

Los coeficientes de variación son elevados en varias variables. Sin embargo, su interpretación debe realizarse con cautela cuando la media se encuentra próxima a cero, como ocurre en `BounceRates`, `ExitRates` y `SpecialDay`.

En `Revenue`, **1.908 de las 12.330 sesiones finalizaron en compra (15,47 %)**, mientras que 10.422 no lo hicieron (84,53 %). Por tanto, la variable objetivo presenta un **desbalance de clases**, aspecto relevante para análisis posteriores.


## Síntesis e interpretación para la toma de decisiones

El análisis exploratorio permite responder preliminarmente la pregunta del proyecto: **las sesiones de comercio electrónico presentan características de navegación que se relacionan con la ocurrencia de una compra**, aunque los resultados no permiten establecer relaciones causales.

Los principales hallazgos son:

- La tasa de conversión observada es **15,47 %**, con IC 95 % de **14,84 % a 16,11 %**.
- `Revenue` está desbalanceada: la mayoría de las sesiones no finaliza en compra.
- Existen relaciones fuertes entre algunas métricas de navegación, particularmente `BounceRates`–`ExitRates` y `ProductRelated`–`ProductRelated_Duration`.
- `VisitorType` presenta una asociación estadísticamente significativa con `Revenue`, aunque su magnitud es baja (**V de Cramér = 0,105**).
- Los nuevos visitantes muestran descriptivamente una mayor proporción de compra que los visitantes recurrentes.
- `PageValues` diferencia claramente a nivel descriptivo las sesiones con y sin compra: media **27,265** y mediana **16,758** en sesiones con compra, frente a media **1,976** y mediana **0** en sesiones sin compra. Debido a su fuerte asimetría, esta diferencia debe analizarse posteriormente con un procedimiento inferencial apropiado.

### Alcance de las conclusiones

Los resultados corresponden a asociaciones observadas en el dataset y constituyen evidencia para orientar análisis posteriores. **No se atribuyen causas** a los patrones detectados. La toma de decisiones debe considerar tanto la significancia estadística como la magnitud de las asociaciones y el contexto del comercio electrónico.

Esta etapa proporciona una base reproducible para profundizar posteriormente en la explicación y modelamiento de la probabilidad de compra.


## 17. Referencias

- Maidana, J. P. (2026). *Tipos de variables y exploración de datos con pandas*. Universidad Andrés Bello.
- Maidana, J. P. (2026). *Bases de datos disponibles para el proyecto del curso*. Universidad Andrés Bello.
- UCI Machine Learning Repository. *Online Shoppers Purchasing Intention Dataset*.
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
- pandas Development Team. *pandas documentation*.
